In [ ]:
# =============================================================
# TAHAP 3: CASE RETRIEVAL
# Mata Kuliah Penalaran Komputer - CBR Korupsi Kerugian Keuangan Negara
# =============================================================

# Tahap 3: Case Retrieval
**Tujuan:** Menemukan kasus lama yang paling mirip dengan query kasus baru
menggunakan TF-IDF + cosine similarity dan model SVM untuk klasifikasi.

In [ ]:
Import Library
import os
import json
import glob
import pickle
import numpy as np
import pandas as pd
from typing import List, Dict, Tuple

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC, LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings("ignore")

In [ ]:
Konfigurasi Path
BASE_DIR  = os.path.dirname(os.path.dirname(os.path.abspath(__file__))) \
            if "__file__" in dir() else os.getcwd()
RAW_DIR   = os.path.join(BASE_DIR, "data", "raw")
PROC_DIR  = os.path.join(BASE_DIR, "data", "processed")
EVAL_DIR  = os.path.join(BASE_DIR, "data", "eval")
MODEL_DIR = os.path.join(BASE_DIR, "models")
os.makedirs(EVAL_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
print(f"✅ Path siap")

In [ ]:
Load Dataset
def load_dataset() -> Tuple[pd.DataFrame, List[str]]:
    """Load dataset dari CSV dan teks mentah."""
    csv_path = os.path.join(PROC_DIR, "cases_full.csv")
    if not os.path.exists(csv_path):
        csv_path = os.path.join(PROC_DIR, "cases.csv")
    df = pd.read_csv(csv_path, encoding="utf-8-sig")
    print(f"✅ Dataset loaded: {len(df)} kasus")

    # Load teks penuh dari file raw
    texts = []
    for case_id in df["case_id"]:
        txt_path = os.path.join(RAW_DIR, f"{case_id}.txt")
        if os.path.exists(txt_path):
            with open(txt_path, "r", encoding="utf-8") as f:
                texts.append(f.read())
        elif "text_full" in df.columns:
            row = df[df["case_id"] == case_id]["text_full"].values
            texts.append(row[0] if len(row) > 0 and pd.notna(row[0]) else "")
        else:
            texts.append(df[df["case_id"] == case_id]["ringkasan_fakta"].values[0] or "")

    df["text"] = texts
    return df, texts

In [ ]:
Preprocessing Teks Bahasa Indonesia
def preprocess_text(text: str) -> str:
    """Preprocessing teks: hapus stopword sederhana, bersihkan."""
    import re
    # Stopword bahasa Indonesia sederhana
    stopwords_id = {
        "yang","dan","di","ke","dari","ini","itu","dengan","untuk","pada",
        "adalah","dalam","tidak","oleh","atau","juga","sudah","akan","ada",
        "bahwa","telah","tersebut","lebih","setelah","serta","dapat","harus",
        "namun","karena","atas","tapi","jika","maka","sebagai","antara","hal",
        "para","bagi","maupun","melalui","berdasarkan","terhadap","sehingga",
        "sebab","yaitu","yakni","akibat","sesuai","selain","secara","selama",
    }
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in stopwords_id and len(t) > 2]
    return " ".join(tokens)

In [ ]:
Bangun TF-IDF Vectorizer
def build_tfidf(texts: List[str], max_features: int = 5000):
    """Bangun dan fit TF-IDF vectorizer."""
    processed = [preprocess_text(t) for t in texts]
    vectorizer = TfidfVectorizer(
        max_features=max_features,
        ngram_range=(1, 2),          # unigram + bigram
        min_df=2,                     # muncul minimal di 2 dokumen
        max_df=0.95,                  # abaikan term yang muncul di >95% dokumen
        sublinear_tf=True,            # log(tf) untuk mengurangi dominasi frekuensi tinggi
    )
    tfidf_matrix = vectorizer.fit_transform(processed)
    print(f"✅ TF-IDF matrix: {tfidf_matrix.shape} (dokumen x fitur)")
    return vectorizer, tfidf_matrix, processed

In [ ]:
Fungsi Utama: retrieve()
# Variabel global untuk diisi saat pipeline dijalankan
_vectorizer    = None
_tfidf_matrix  = None
_case_ids      = None
_df_global     = None

def retrieve(query: str, k: int = 5) -> List[str]:
    """
    Fungsi CBR Retrieval utama.
    Args:
        query : Teks deskripsi kasus baru
        k     : Jumlah kasus paling mirip yang dikembalikan
    Returns:
        List of case_id (top-k)
    """
    global _vectorizer, _tfidf_matrix, _case_ids

    if _vectorizer is None:
        raise ValueError("Model belum diinisialisasi. Jalankan run_retrieval_pipeline() dulu.")

    # 1) Preprocessing query
    processed_query = preprocess_text(query)

    # 2) Hitung vektor query menggunakan TF-IDF yang sudah di-fit
    query_vec = _vectorizer.transform([processed_query])

    # 3) Hitung cosine similarity dengan semua vektor kasus
    similarities = cosine_similarity(query_vec, _tfidf_matrix).flatten()

    # 4) Ambil top-k indeks dengan similarity tertinggi
    top_k_indices = similarities.argsort()[::-1][:k]
    top_k_cases   = [(
        _case_ids[i],
        float(similarities[i])
    ) for i in top_k_indices]

    return top_k_cases   # [(case_id, similarity_score), ...]

In [ ]:
Training Model SVM / Naive Bayes untuk Klasifikasi
def train_classifier(df: pd.DataFrame, processed_texts: List[str],
                     method: str = "svm") -> dict:
    """
    Melatih model klasifikasi (SVM atau Naive Bayes) pada label putusan.
    method: 'svm' | 'naive_bayes'
    """
    print(f"\n[Training Classifier - {method.upper()}]")

    labels       = df["label"].values
    le           = LabelEncoder()
    y_encoded    = le.fit_transform(labels)

    # Pastikan minimal ada 2 label
    unique_labels = np.unique(y_encoded)
    if len(unique_labels) < 2:
        print("  ⚠️  Hanya ada 1 kelas label — tambahkan variasi data.")
        print("     Menggunakan dummy classifier sebagai fallback.")
        return {"model": None, "label_encoder": le, "accuracy": 0.0}

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        range(len(processed_texts)), y_encoded,
        test_size=0.2, random_state=42, stratify=y_encoded
    )
    vectorizer_cls = TfidfVectorizer(max_features=3000, ngram_range=(1,2), sublinear_tf=True)

    X_train_texts = [processed_texts[i] for i in X_train]
    X_test_texts  = [processed_texts[i] for i in X_test]
    X_tr_vec = vectorizer_cls.fit_transform(X_train_texts)
    X_te_vec = vectorizer_cls.transform(X_test_texts)

    # Pilih model
    if method == "naive_bayes":
        model = MultinomialNB(alpha=0.1)
    else:  # default SVM
        model = LinearSVC(C=1.0, max_iter=2000, random_state=42)

    model.fit(X_tr_vec, y_train)
    y_pred = model.predict(X_te_vec)
    acc    = accuracy_score(y_test, y_pred)

    print(f"  Accuracy  : {acc:.4f}")
    print(f"\n  Classification Report:")
    print(classification_report(y_test, y_pred,
                                 target_names=le.classes_,
                                 zero_division=0))

    # Simpan model
    model_path = os.path.join(MODEL_DIR, f"classifier_{method}.pkl")
    with open(model_path, "wb") as f:
        pickle.dump({"model": model, "vectorizer": vectorizer_cls, "label_encoder": le}, f)
    print(f"  Model tersimpan: {model_path}")

    return {
        "model"         : model,
        "vectorizer_cls": vectorizer_cls,
        "label_encoder" : le,
        "accuracy"      : acc,
        "X_test"        : X_te_vec,
        "y_test"        : y_test,
        "y_pred"        : y_pred,
    }

In [ ]:
Buat Query Uji (ground-truth)
def create_eval_queries(df: pd.DataFrame, n_queries: int = 8) -> List[Dict]:
    """
    Membuat file queries.json untuk evaluasi retrieval.
    Ambil sebagian kasus sebagai "query baru" dengan ground-truth diketahui.
    """
    # Gunakan kasus dari test set sebagai query
    sample = df.sample(n=min(n_queries, len(df)), random_state=42).reset_index(drop=True)
    queries = []
    for _, row in sample.iterrows():
        queries.append({
            "query_id"       : f"q_{row['case_id']}",
            "query_text"     : str(row["ringkasan_fakta"])[:500],
            "ground_truth_id": row["case_id"],
            "label"          : row["label"],
            "terdakwa"       : row.get("terdakwa", ""),
        })

    path = os.path.join(EVAL_DIR, "queries.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(queries, f, ensure_ascii=False, indent=2)
    print(f"\n✅ {len(queries)} query uji tersimpan: {path}")
    return queries

In [ ]:
Uji Fungsi retrieve()
def test_retrieve(queries: List[Dict], k: int = 5):
    """Uji fungsi retrieve() dengan query yang sudah dibuat."""
    print(f"\n{'='*60}")
    print("  UJI FUNGSI retrieve()")
    print(f"{'='*60}")
    hits = 0
    for q in queries:
        results = retrieve(q["query_text"], k=k)
        retrieved_ids = [r[0] for r in results]
        hit = q["ground_truth_id"] in retrieved_ids
        if hit:
            hits += 1
        print(f"\n  Query   : {q['query_id']}")
        print(f"  G-Truth : {q['ground_truth_id']}")
        print(f"  Top-{k}  : {retrieved_ids}")
        print(f"  {'✅ HIT' if hit else '❌ MISS'} | Skor teratas: {results[0][1]:.4f}")

    recall_at_k = hits / len(queries)
    print(f"\n  Recall@{k}: {recall_at_k:.4f} ({hits}/{len(queries)} hit)")

In [ ]:
Pipeline Utama
def run_retrieval_pipeline():
    global _vectorizer, _tfidf_matrix, _case_ids, _df_global
    print("\n" + "="*60)
    print("  TAHAP 3: CASE RETRIEVAL")
    print("="*60)

    # Load data
    print("\n[1/5] Load dataset...")
    df, texts = load_dataset()
    _df_global = df
    _case_ids  = df["case_id"].tolist()

    # Bangun TF-IDF
    print("\n[2/5] Membangun TF-IDF vectorizer...")
    _vectorizer, _tfidf_matrix, processed_texts = build_tfidf(texts)

    # Simpan vectorizer & matrix
    with open(os.path.join(MODEL_DIR, "tfidf_vectorizer.pkl"), "wb") as f:
        pickle.dump(_vectorizer, f)
    np.save(os.path.join(MODEL_DIR, "tfidf_matrix.npy"), _tfidf_matrix.toarray())
    with open(os.path.join(MODEL_DIR, "case_ids.json"), "w") as f:
        json.dump(_case_ids, f)
    print("  Model TF-IDF tersimpan.")

    # Training SVM
    print("\n[3/5] Training model SVM...")
    svm_result = train_classifier(df, processed_texts, method="svm")

    # Training Naive Bayes
    print("\n[4/5] Training model Naive Bayes...")
    nb_result = train_classifier(df, processed_texts, method="naive_bayes")

    # Buat query uji & test retrieve
    print("\n[5/5] Membuat query evaluasi & uji retrieve()...")
    queries = create_eval_queries(df, n_queries=8)
    test_retrieve(queries, k=5)

    print("\n✅ Tahap 3 selesai!")
    print(f"   SVM Accuracy : {svm_result['accuracy']:.4f}")
    print(f"   NB  Accuracy : {nb_result['accuracy']:.4f}")
    print("="*60)

    return {
        "vectorizer"    : _vectorizer,
        "tfidf_matrix"  : _tfidf_matrix,
        "case_ids"      : _case_ids,
        "svm_result"    : svm_result,
        "nb_result"     : nb_result,
    }

In [ ]:
Fungsi Load Model (untuk notebook lain)
def load_retrieval_model():
    """Load model retrieval yang sudah disimpan."""
    global _vectorizer, _tfidf_matrix, _case_ids
    with open(os.path.join(MODEL_DIR, "tfidf_vectorizer.pkl"), "rb") as f:
        _vectorizer = pickle.load(f)
    _tfidf_matrix = np.load(os.path.join(MODEL_DIR, "tfidf_matrix.npy"))
    from scipy.sparse import csr_matrix
    _tfidf_matrix = csr_matrix(_tfidf_matrix)
    with open(os.path.join(MODEL_DIR, "case_ids.json"), "r") as f:
        _case_ids = json.load(f)
    print(f"✅ Model loaded | {len(_case_ids)} kasus")

if __name__ == "__main__":
    results = run_retrieval_pipeline()